# STARE-PODS cloud ingest — SDK demo (10 granules)

End-to-end walkthrough of the Path C client SDK: submit an ingest job to the
deployed REST API, wait for it to finish, and inspect the results.

Deliberately scoped to **10 granules** so it finishes in ~5–6 minutes (the full
load test uses 69). With `workers=4` the scheduler makes `ceil(10/4)=3` → tickets
`[3,3,3,1]`, i.e. all 4 workers in parallel, ~3 granules each.

**Prerequisites**
- `conda activate starepandas_3.12_v3`
- The cloud `endpoint` + `api_key` are read from `starepandas/.config` (or the
  `STAREPANDAS_CLOUD_ENDPOINT` / `STAREPANDAS_CLOUD_API_KEY` env vars). The key
  is **never printed** by this notebook.
- The input granules are already staged in S3 (see the URIs below).

Operations reference: `docs/path_c_runbook.md`.

In [1]:
import os

# Point the config loader at the project .config (holds endpoint + api_key).
# Adjust the path if you run this notebook from a different working directory.
os.environ.setdefault("STAREPANDAS_AWS_CONFIG", os.path.abspath("../starepandas/.config"))

import starepandas as sp

# Confirm the SDK can resolve the API endpoint (the key value is not shown).
endpoint, _api_key = sp.cloud.get_cloud_config()
print("cloud endpoint:", endpoint)
print("api_key resolved:", bool(_api_key))

cloud endpoint: https://yujvpbbs7j.execute-api.us-west-2.amazonaws.com/v1
api_key resolved: True


## 1. Pick the inputs

Ten staged SSMIS granules (a job is single-instrument). Hardcoded so the
notebook needs no S3-list permissions.

Orbits **078441** and **078446** are deliberately excluded: the example
demos ingest those same granules to `gmi-demo-parquet`, and a catalog row
(`pods_unique` keys on dataset + collected time + `grouped_id`, not the
output prefix) can only point at **one** `group_path` — two jobs ingesting
the same granule to different prefixes silently steal each other's rows,
whichever ran last winning. Disjoint inputs make the ordering irrelevant.

In [2]:
INSTRUMENT = "SSMIS"
BASE = "s3://zarrpods/testing-s3/loadtest-jan/input"
granule_uris = [
    f"{BASE}/1C.F18.SSMIS.XCAL2021-V.20250101-S011702-E025853.078435.V07B.HDF5",
    f"{BASE}/1C.F18.SSMIS.XCAL2021-V.20250101-S025854-E044045.078436.V07B.HDF5",
    f"{BASE}/1C.F18.SSMIS.XCAL2021-V.20250101-S044046-E062236.078437.V07B.HDF5",
    f"{BASE}/1C.F18.SSMIS.XCAL2021-V.20250101-S080429-E094620.078439.V07B.HDF5",
    f"{BASE}/1C.F18.SSMIS.XCAL2021-V.20250101-S094621-E112812.078440.V07B.HDF5",
        f"{BASE}/1C.F18.SSMIS.XCAL2021-V.20250101-S131005-E145155.078442.V07B.HDF5",
    f"{BASE}/1C.F18.SSMIS.XCAL2021-V.20250101-S145156-E163347.078443.V07B.HDF5",
    f"{BASE}/1C.F18.SSMIS.XCAL2021-V.20250101-S163348-E181539.078444.V07B.HDF5",
    f"{BASE}/1C.F18.SSMIS.XCAL2021-V.20250101-S181540-E195731.078445.V07B.HDF5",
    f"{BASE}/1C.F18.SSMIS.XCAL2021-V.20250101-S213924-E232114.078447.V07B.HDF5",
]
OUT_PREFIX = "s3://zarrpods/testing-s3/loadtest-jan/storage"
print(f"{len(granule_uris)} {INSTRUMENT} granules")

10 SSMIS granules


## 1.5 (Optional) Start fresh — clean a previous run

Re-running the demo does **not** create duplicate RDS rows: chunk metadata in
`PodsMetadata` is keyed by `grouped_id` (pod-code + granule + dataset, *not* the
output prefix), so the writes are an **idempotent upsert** (§C10 #1) — the same
rows are updated in place.

The **S3 side** is where stale data accumulates:
- **Same `OUT_PREFIX`** → keys are deterministic (`…/q<pod>-<granule>-<dataset>.parquet`),
  so a re-run **overwrites** them — no growth.
- **Changed `OUT_PREFIX`** between runs → a fresh copy lands under the new prefix
  and the **old objects are orphaned** (the upserted RDS rows now point at the new
  prefix, but the old S3 objects are not auto-deleted).

`clean_s3_prefix(OUT_PREFIX)` wipes both sides under the prefix — every S3 object
**and** every `PodsMetadata` row whose `group_path` falls under it — so each run
starts from a clean slate. (It targets the `StarePodsMetadata` RDS database that
the cloud worker writes to.)

In [3]:
# Set False to keep prior data (re-run will overwrite same keys / orphan changed prefixes).
CLEAN_BEFORE_RUN = True

if CLEAN_BEFORE_RUN:
    cleaned = sp.clean_s3_prefix(OUT_PREFIX)
    print(f"cleaned {OUT_PREFIX}")
    print(f"  RDS PodsMetadata rows deleted: {cleaned['rds_rows_deleted']}")
    print(f"  S3 objects deleted:            {cleaned['s3_objects_deleted']}")
else:
    print("CLEAN_BEFORE_RUN = False — skipping cleanup")

cleaned s3://zarrpods/testing-s3/loadtest-jan/storage
  RDS PodsMetadata rows deleted: 12756
  S3 objects deleted:            14225


## 2. Submit the job

`ingest_granules(...)` POSTs to `/ingest` and returns a `JobHandle`. The `202`
response is cached on `handle.record`.

> Each POST mints a **new** `job_id` — don't blindly re-run this cell to "retry";
> it would submit a second job.

In [4]:
handle = sp.cloud.ingest_granules(
    granule_uris=granule_uris,
    instrument=INSTRUMENT,
    workers=4,                 # default 4; hard cap 4
    s3_prefix=OUT_PREFIX,      # optional; isolates demo output
)
print("job_id:", handle.job_id)
handle.record   # 202 body: state=running, total_granules, ticket_count, workers

job_id: 35422080-6fb6-435a-a18c-cf45911c4b2e


{'job_id': '35422080-6fb6-435a-a18c-cf45911c4b2e',
 'state': 'running',
 'total_granules': 10,
 'ticket_count': 4,
 'workers': 4}

## 3. Wait for completion

`wait()` polls `GET /jobs/{id}` until the state is terminal (`complete` /
`failed`). Cold start is ~2 min, then ~3 granules per worker.

In [5]:
from datetime import datetime

record = handle.wait(poll_interval=15)
print("state:", record["state"])
print("processed / total:", record.get("processed"), "/", record.get("total_granules"))
print("failed:", record.get("failed"))
print("ticket_count:", record.get("ticket_count"), "workers:", record.get("workers"))
print("created_at:", record.get("created_at"))
print("completed_at:", record.get("completed_at"))

# Total wall-clock the job took (completed_at - created_at), so you don't have to
# subtract by hand. Both are ISO-8601 UTC strings from the API.
created, completed = record.get("created_at"), record.get("completed_at")
if created and completed:
    elapsed = (datetime.fromisoformat(completed) - datetime.fromisoformat(created)).total_seconds()
    print(f"elapsed: {elapsed:.0f} s  ({int(elapsed // 60)}m {int(elapsed % 60)}s)")

assert record["state"] == "complete" and record.get("processed") == len(granule_uris)

state: complete
processed / total: 10 / 10
failed: 0
ticket_count: 4 workers: 4
created_at: 2026-08-21T21:41:31.172523+00:00
completed_at: 2026-08-21T21:47:05.029352+00:00
elapsed: 334 s  (5m 33s)


## 4. Inspect the results

`handle.failures()` returns the **per-granule ledger** from `StarePodsFailures`:
one row per granule with its outcome (`state="processed"` on success — these rows
also power idempotent dedupe of redelivered tickets). *Actual* failures are the
rows whose state isn't `processed`; for a clean run that's **0**. Then we count
the Parquet objects this job wrote (scoped to its own granules — the storage
prefix is shared across runs) and read one partition back to show the data
round-trips.

In [6]:
# Per-granule ledger: StarePodsFailures records EVERY granule's outcome
# (state="processed" on success); real failures are rows whose state != "processed".
ledger = handle.failures()
rows = ledger.get("failures", [])
real_failures = [r for r in rows if r.get("state") != "processed"]
print("per-granule ledger rows:", ledger.get("count"), "| actual failures:", len(real_failures))

# Count the Parquet objects THIS job wrote, scoped to its own granule basenames
# (the storage prefix is shared with other runs, so an unscoped count is cumulative).
import boto3
cfg = {}
for line in open(os.environ["STAREPANDAS_AWS_CONFIG"]):
    s = line.strip()
    if "=" in s and not s.startswith("#"):
        k, v = s.split("=", 1)
        cfg[k.strip()] = v.strip()
s3 = boto3.Session(
    aws_access_key_id=cfg["key"], aws_secret_access_key=cfg["secret"],
    region_name=cfg.get("region_name", "us-west-2"),
).client("s3")

basenames = [u.split("/")[-1].replace(".HDF5", "") for u in granule_uris]
bucket = OUT_PREFIX.split("/")[2]
prefix = OUT_PREFIX.split(bucket + "/", 1)[1]
n, sample_key = 0, None
for page in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
    for o in page.get("Contents", []):
        if o["Key"].endswith(".parquet") and any(b in o["Key"] for b in basenames):
            n += 1
            sample_key = sample_key or o["Key"]
print(f"Parquet objects for this job's {len(granule_uris)} granules: {n} (~{n // len(granule_uris)}/granule)")
print("sample partition:", sample_key)

per-granule ledger rows: 10 | actual failures: 0


Parquet objects for this job's 10 granules: 14248 (~1424/granule)
sample partition: testing-s3/loadtest-jan/storage/q000000-1C.F18.SSMIS.XCAL2021-V.20250101-S011702-E025853.078435.V07B-SSMIS_S1.parquet


In [7]:
# Read one partition back to confirm real data landed.
import pandas as pd
df = pd.read_parquet(
    f"s3://{bucket}/{sample_key}",
    storage_options={"key": cfg["key"], "secret": cfg["secret"]},
)
print("rows:", len(df), " columns:", list(df.columns)[:8], "...")
df.head()

rows: 599  columns: ['lat', 'lon', 'sids', 'timestamp', 'Tc1', 'Tc2', 'Tc3', '__row_positions__'] ...


,lat,lon,sids,timestamp,Tc1,Tc2,Tc3,__row_positions__
0,-26.760332,14.764282,1126985244781513,2025-01-01 02:38:38.282,188.710007,117.559998,206.250000,232065
1,-26.870766,14.734711,1134380463222281,2025-01-01 02:38:40.178,188.279999,116.220001,204.860001,232155
2,-26.981190,14.705078,1132554606299273,2025-01-01 02:38:42.074,187.080002,115.449997,203.929993,232245
3,-27.044901,14.417236,1142369975341481,2025-01-01 02:38:43.974,188.080002,116.099998,206.559998,232334
4,-27.091892,14.675323,1155255574996297,2025-01-01 02:38:43.974,187.410004,115.519997,204.580002,232335


## Wrap-up

- 10 granules typically finish in ~5–6 min (cold start + ~3 granules/worker) and
  cost ~pennies; the watcher then scales the workers back to 0 automatically.
- Other handle methods: `handle.status()` (single GET), `handle.failures(next_token=...)`
  (paginated per-granule ledger), `handle.cancel()` — which raises
  `NotImplementedError` (the API returns `501`; cancellation is deferred in v1).
- For operations — stuck jobs, secret rotation, image rollback, error→cause — see
  `docs/path_c_runbook.md`.